In [ ]:
# # Ultralytics YOLO es una biblioteca de código abierto que implementa el modelo de detección de objetos 
# YOLO (You Only Look Once). Es ampliamente utilizada para tareas de visión por computadora, como la detección 
# de objetos en imágenes y videos. En este proyecto, utilizaremos Ultralytics YOLO para detectar siniestros 
# en las imágenes proporcionadas.
try:
    import ultralytics
    print(f"¡Todo listo! Ultralytics YOLO versión {ultralytics.__version__} está instalada y lista para usarse.")
except ImportError:
    print("La librería 'ultralytics' no está instalada. Ejecuta: !pip install ultralytics")

¡Todo listo! Ultralytics YOLO versión 8.4.60 está instalada y lista para usarse.


In [ ]:
try:
    import ultralytics # Ultralytics YOLO es una biblioteca de código abierto que implementa el modelo de detección de objetos YOLO (You Only Look Once). Es ampliamente utilizada para tareas de visión por computadora, como la detección de objetos en imágenes y videos. En este proyecto, utilizaremos Ultralytics YOLO para detectar siniestros en las imágenes proporcionadas.
    print(f"¡Todo listo! Ultralytics YOLO versión {ultralytics.__version__} está instalada y lista para usarse.")
except ImportError:
    print("La librería 'ultralytics' no está instalada. Ejecuta: !pip install ultralytics")

¡Todo listo! Ultralytics YOLO versión 8.4.60 está instalada y lista para usarse.


🧾 BLOQUE 1: El Script de Captura y Procesamiento

In [27]:
# =====================================================================
# LIBRERÍAS (Módulos que aportan las funciones clave al script)
# =====================================================================

# os (Operating System): Permite gestionar carpetas, listar archivos y unir rutas del disco.
import os

# json (JavaScript Object Notation): Se encarga de estructurar, leer y guardar archivos .json.
import json

# YOLO (You Only Look Once): Descarga y gestiona la red neuronal convolucional de visión artificial.
from ultralytics import YOLO

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS ABSOLUTAS Y CARGA DEL MODELO DE IA
# =====================================================================

# CARPETA_RAIZ_INPUT: Ruta de origen. La 'r' inicial (raw string) evita que Windows confunda las barras '\' con comandos de texto.
CARPETA_RAIZ_INPUT = r"C:\Users\Carlos\Documents\Curso_Analisis_Data_bootcamp_Upgrade_Hub\Redes_Neuronales\verificacion_siniestros\data\input_imagenes"

# CARPETA_OUTPUT: Ruta real física en tu ordenador donde se guardarán los expedientes JSON resultantes.
CARPETA_OUTPUT = r"C:\Users\Carlos\Documents\Curso_Analisis_Data_bootcamp_Upgrade_Hub\Redes_Neuronales\verificacion_siniestros\data\out_put_json"

# os.path.join: Une los componentes para localizar el archivo del modelo de forma compatible con el sistema operativo.
RUTA_MODELO = os.path.join("modelos", "yolov8n.pt")

# model: Instanciamos el modelo cargándolo en la memoria RAM una sola vez fuera del bucle para ahorrar recursos y tiempo.
model = YOLO(RUTA_MODELO)

# =====================================================================
# 2. DETECCIÓN AUTOMÁTICA DE EXPEDIENTES EN EL INPUT
# =====================================================================

# List Comprehension: Escanea la raíz del input, une las rutas y filtra descartando archivos sueltos.
# os.path.isdir() garantiza que solo se guarden en la lista los nombres de las subcarpetas físicas (los siniestros).
siniestros = [f for f in os.listdir(CARPETA_RAIZ_INPUT) if os.path.isdir(os.path.join(CARPETA_RAIZ_INPUT, f))]

# Imprimimos en la consola de Jupyter un resumen inicial del volumen de trabajo detectado.
print(f"🤖 Se han detectado {len(siniestros)} siniestros para procesar.\n")

# =====================================================================
# 3. BUCLE PRINCIPAL: PROCESAMIENTO SECUENCIAL POR EXPEDIENTE (BATCH)
# =====================================================================

# El bucle 'for' procesará un expediente de siniestro de manera independiente a la vez.
for id_siniestro in siniestros:
    print(f"--------------------------------------------------")
    print(f"📦 PROCESANDO EXPEDIENTE: {id_siniestro}")
    print(f"--------------------------------------------------")
    
    # Construimos la ruta hacia la subcarpeta del siniestro que se está evaluando en esta vuelta del bucle.
    carpeta_siniestro_actual = os.path.join(CARPETA_RAIZ_INPUT, id_siniestro)
    
    # Inicializamos la estructura de datos limpia del expediente en memoria RAM.
    # Al declararlo aquí dentro, nos aseguramos de vaciar el diccionario en cada iteración del bucle.
    expediente_final = {
        "id_siniestro": id_siniestro,
        "matricula_vehiculo": "PENDIENTE_REVISION",          # Estado base inicial
        "importe_reparacion": 0.0,                            # Campo económico que llenaremos más abajo
        "usuario_responsable": "PERITO_CARLOS_GOMEZ",         # Firma digital de gobierno de datos y trazabilidad
        "camino_a_automatizado": [],                          # Lista vacía que guardará metadatos de fotos generales
        "camino_b_revision_danos": []                         # Lista vacía que guardará metadatos de fotos de golpes
    }
    
    # Buscamos y filtramos los archivos de imágenes válidos dentro de la subcarpeta del siniestro actual.
    # .lower().endswith() convierte el nombre a minúsculas para validar extensiones sin errores de tipografía.
    imagenes = [f for f in os.listdir(carpeta_siniestro_actual) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
    
    # Recorremos cada una de las imágenes del expediente actual de forma secuencial.
    for img_nombre in imagenes:
        # Construimos la ruta absoluta de la imagen a analizar.
        ruta_imagen = os.path.join(carpeta_siniestro_actual, img_nombre)
        
        # Inferencia: Pasamos la foto por la red neuronal YOLO. verbose=False apaga las alertas nativas de la IA en consola.
        resultados = model(ruta_imagen, verbose=False)
        
        # YOLO devuelve una lista. Extraemos el primer índice [0] que contiene los objetos detectados (boxes) de la foto.
        resultado_img = resultados[0]
        
        # Flag o Bandera lógica: Actúa como un interruptor para desviar la foto al camino correspondiente.
        paso_automatizado = False
        
        # Recorremos todas las cajas de detección que la IA ha encontrado dentro de la foto actual.
        for box in resultado_img.boxes:
            # Extraemos el ID de la clase detectada (ej: 2) y lo transformamos a un entero nativo de Python.
            clase_id = int(box.cls[0])
            # Buscamos en el diccionario del modelo el nombre asociado a ese ID (ej: 2 corresponde a 'car').
            clase_nombre = model.names[clase_id]
            # Extraemos el nivel de confianza de la predicción de la IA (un número decimal entre 0.0 y 1.0).
            confianza = float(box.conf[0])
            
            # REGLA DE NEGOCIO: Si el objeto detectado es un coche entero y la IA tiene un 60% o más de seguridad...
            if clase_nombre == "car" and confianza >= 0.60:
                # Insertamos los metadatos de la imagen en la lista del CAMINO A (Flujo automatizado)
                expediente_final["camino_a_automatizado"].append({
                    "archivo": img_nombre,
                    "analisis": "Vehículo verificado con éxito",
                    "confianza_ia": f"{confianza * 100:.2f}%"  # Convertimos a porcentaje con dos decimales de lectura simple
                })
                paso_automatizado = True  # Activamos el interruptor indicando que la foto superó el filtro
                print(f"   🚗 CAMINO A -> [{img_nombre}] Coche verificado")
                break  # Rompemos este bucle interno. Si ya encontramos el coche entero, no perdemos tiempo buscando más objetos en la misma foto.
                
        # CRITERIO DE EXCEPCIÓN: Si se revisaron todos los objetos y la bandera sigue en False (plano cerrado, pieza rota o sin coche)...
        if not paso_automatizado:
            # Desviamos los metadatos de la imagen a la lista del CAMINO B (Revisión manual de daños por el analista)
            expediente_final["camino_b_revision_danos"].append({
                "archivo": img_nombre,
                "analisis": "Evidencia de daño o plano cercano"
            })
            print(f"   🔍 CAMINO B -> [{img_nombre}] Guardada para revisión de daños")

    # =====================================================================
    # 4. INTERVENCIÓN MANUAL CONSOLIDADA (Matrícula + Presupuesto en Caliente)
    # =====================================================================
    print(f"\n   ⚠️ CONTROL PERICIAL DE SEGURIDAD para {id_siniestro}:")
    
    # Captura de Matrícula: input() detiene el script y espera la escritura por teclado del operador.
    matricula_manual = input(f"   ✍ * [1/2] Introduzca la matrícula para {id_siniestro}: ")
    # .strip() elimina espacios accidentales y .upper() normaliza el texto a mayúsculas para la base de datos.
    if matricula_manual.strip():
        expediente_final["matricula_vehiculo"] = matricula_manual.strip().upper()
    
    # Captura de Presupuesto: Usamos una estructura de control defensiva contra errores de escritura del usuario.
    presupuesto_valido = False
    while not presupuesto_valido:
        presupuesto_manual = input(f"   💰 [2/2] Introduzca el Presupuesto Inicial (€) para {id_siniestro}: ")
        try:
            # .replace(",", ".") cambia el formato decimal europeo a inglés para evitar que float() se rompa.
            monto_float = float(presupuesto_manual.strip().replace(",", "."))
            # Asignamos el valor numérico limpio directamente al diccionario final
            expediente_final["importe_reparacion"] = monto_float
            presupuesto_valido = True # Cambiamos el estado a True para romper el bucle infinito del while
        except ValueError:
            # Si el usuario introduce letras o caracteres extraños, capturamos el fallo sin colapsar el programa.
            print("      ❌ Error: Por favor, introduzca un monto numérico válido (ej: 1250.75).")
            
    # =====================================================================
    # 5. GUARDADO FÍSICO Y PERSISTENCIA EN EL DISCO DURO
    # =====================================================================
    
    # Red de seguridad: Si la carpeta de destino 'out_put_json' no existe en Windows, la creamos al vuelo.
    if not os.path.exists(CARPETA_OUTPUT):
        os.makedirs(CARPETA_OUTPUT)
        
    # Construimos el nombre definitivo del archivo JSON basado en el siniestro actual.
    ruta_json_siniestro = os.path.join(CARPETA_OUTPUT, f"{id_siniestro}_reporte.json")
    
    # open() abre el canal en modo "w" (Write / Escritura). encoding="utf-8" asegura la integridad de acentos y la 'ñ'.
    with open(ruta_json_siniestro, "w", encoding="utf-8") as archivo:
        # json.dump escribe el diccionario en el archivo físico. indent=4 tabula el texto de forma estética.
        # ensure_ascii=False obliga a guardar caracteres latinos reales en vez de códigos Unicode extraños.
        json.dump(expediente_final, archivo, indent=4, ensure_ascii=False)
        
    print(f"\n💾 ¡Expediente {id_siniestro} firmado y guardado con éxito!")

# Cierre definitivo del script masivo
print("\n🚀 --- PROCESO MASIVO COMPLETADO ---")

🤖 Se han detectado 2 siniestros para procesar.

--------------------------------------------------
📦 PROCESANDO EXPEDIENTE: SIN-2026-001
--------------------------------------------------
   🔍 CAMINO B -> [Imagen0.jpeg] Guardada para revisión de daños
   🔍 CAMINO B -> [Imagen2.jpeg] Guardada para revisión de daños

   ⚠️ CONTROL PERICIAL DE SEGURIDAD para SIN-2026-001:

💾 ¡Expediente SIN-2026-001 firmado y guardado con éxito!
--------------------------------------------------
📦 PROCESANDO EXPEDIENTE: SIN-2026-002
--------------------------------------------------
   🚗 CAMINO A -> [Imagen1.jpeg] Coche verificado
   🔍 CAMINO B -> [Imagen3.jpeg] Guardada para revisión de daños

   ⚠️ CONTROL PERICIAL DE SEGURIDAD para SIN-2026-002:

💾 ¡Expediente SIN-2026-002 firmado y guardado con éxito!

🚀 --- PROCESO MASIVO COMPLETADO ---


🧾 PARTE 1: El Script de Captura y Procesamiento (Jupyter)
Este script funciona como una línea de ensamblaje secuencial masiva (Batch Processing). Su objetivo es procesar n carpetas de siniestros de forma independiente sin saturar la memoria RAM.

1. Inicialización y Carga de Infraestructura

- import os / import json / from ultralytics import YOLO: Cargamos las tres cajas de herramientas necesarias. Es una práctica estándar poner las importaciones al inicio para que Python reserve el espacio correspondiente en memoria antes de ejecutar cualquier lógica.

- Rutas Absolutas con prefijo r"": Al usar la r antes de las comillas, le indicamos a Python que trate el texto como una cadena "cruda" (raw string). En Windows, los directorios usan barras invertidas (\). Si omitiéramos la r, Python intentaría interpretar combinaciones como \n o \v como saltos de línea o tabulaciones verticales, rompiendo la ruta del disco.

- model = YOLO(os.path.join("modelos", "yolov8n.pt")): Instanciamos el modelo. Este paso es crítico situarlo fuera del bucle principal. Cargar un modelo de red neuronal en la memoria RAM (o VRAM si usas tarjeta gráfica) es una operación computacionalmente costosa que toma unos segundos. Al inicializarlo arriba, el modelo se carga una sola vez y se queda latente, listo para recibir imágenes en ráfaga.

2. El Escáner de Entrada (Input Discovery)
La List Comprehension de siniestros: ```python
siniestros = [f for f in os.listdir(CARPETA_RAIZ_INPUT) if os.path.isdir(os.path.join(CARPETA_RAIZ_INPUT, f))]

Esta línea lee la carpeta raíz (`data/input_imagenes`). `os.listdir` devuelve una lista de *todo* lo que hay dentro (tanto archivos sueltos como carpetas). Para evitar que un archivo basura (como un `.DS_Store` o un `.txt` perdido) rompa el flujo, aplicamos el filtro condicional `os.path.isdir()`. Este comando verifica si el elemento es estrictamente una subcarpeta. Al final, la variable `siniestros` contiene una lista limpia de cadenas de texto: `['SIN-2026-001', 'SIN-2026-002']`.

3. El Bucle Principal (For Loop por Expediente)

- for id_siniestro in siniestros: Iniciamos la iteración secuencial. Todo lo que esté indentado debajo de este bucle se ejecutará por separado para cada expediente detectado.

- Aislamiento y Reinicio de Variables: Al inicio del bucle definimos expediente_final como un diccionario nativo de Python con campos preestablecidos (como "importe_reparacion": 0.0 y la constante "usuario_responsable": "PERITO_CARLOS_GOMEZ"). Al declararlo aquí dentro, garantizamos que en cada vuelta del bucle el diccionario anterior se destruye y se genera uno completamente nuevo y limpio, evitando que los datos del siniestro anterior se mezclen con el actual.

4. Filtrado del Embudo de Imágenes (Lógica del Negocio)
- Detección de extensiones: f.lower().endswith(...) se asegura de listar únicamente archivos visuales procesables por el modelo informático, previniendo errores de lectura.
- resultados = model(ruta_imagen, verbose=False): Pasamos la imagen por las capas convolucionales de la IA. El parámetro verbose=False es estético: suprime los textos por defecto de YOLO en la consola para que el perito solo vea los mensajes personalizados que tú programaste.
- Estructura de Extracción de YOLO: El modelo devuelve una lista de objetos tipo Results. Como procesamos las imágenes individualmente, extraemos el primer índice con resultados[0]. Dentro de este objeto, resultado_img.boxes contiene las coordenadas y metadatos de todos los "objetos o cajas de detección" encontrados.
- El bucle de cajas (for box in resultado_img.boxes:):
- box.cls[0] devuelve el ID numérico de la clase detectada (por ejemplo, el número 2 para coches). Se convierte a int para indexarlo en la lista interna de nombres del modelo (model.names[clase_id]), recuperando el texto string nativo ('car').
- box.conf[0] extrae la probabilidad o nivel de confianza (un decimal flotante entre 0.0 y 1.0).
- La Bandera de Control (paso_automatizado = False): Funciona como un interruptor lógico. Si la IA detecta que el objeto es un 'car' y su certeza es igual o superior al 60% (0.60), añade la metadata estructurada al Camino A, cambia el interruptor a True y ejecuta un break. Este break detiene de inmediato el análisis de esa foto en particular; si ya confirmamos que la foto es una toma general válida del coche, no tiene sentido perder ciclos de procesamiento revisando si hay otros objetos secundarios en el fondo.
- El Criterio de Excepción: Si el bucle de cajas termina y la bandera sigue en False (lo que significa que la foto es un plano cerrado del daño, una pieza suelta o una imagen con baja confianza), el bloque if not paso_automatizado: captura la foto y la desvía de forma segura al Camino B para su peritaje visual en la oficina.

5. Intervención Manual Consolidada (Human-in-the-Loop)
- input(): Congela temporalmente el hilo de ejecución del procesador. El sistema operativo espera a que el operador introduzca una cadena de caracteres por teclado.

- matricula_manual.strip().upper(): .strip() elimina posibles espacios en blanco involuntarios al inicio o al final del texto introduced por el perito. .upper() estandariza las letras a mayúsculas para cumplir con los requisitos de normalización de cualquier base de datos relacional.

- El Bucle Antierrores (while not presupuesto_valido:): Es una estructura de control de excepciones defensiva. Si el usuario ingresa texto inválido (por ejemplo: "mil euros" en lugar de 1000), la función float() fallará lanzando un error de tipo ValueError. El bloque except ValueError: atrapa ese fallo, evita que el programa se cierre abruptamente, muestra un aviso de advertencia en pantalla y vuelve a solicitar el dato de forma indefinida hasta que el valor sea un número real procesable por el sistema. El método .replace(",", ".") es una ayuda de usabilidad para transformar el formato decimal europeo al estándar informático anglosajón.

6. Persistencia de Datos en Disco Duro
- os.makedirs(CARPETA_OUTPUT): Si es la primera vez que se ejecuta el script y la carpeta out_put_json no existe en Windows, este comando la crea dinámicamente.

- with open(...) as archivo: Abre un canal de escritura de archivos (File Stream) en modo "w" (Write). Al usar la sentencia with, Python se encarga de cerrar de forma automática y segura el archivo en el disco una vez que termine la escritura, previniendo la corrupción de datos o bloqueos de archivos en el sistema operativo.

- json.dump(..., indent=4, ensure_ascii=False): Convierte el diccionario dinámico de la memoria RAM a un formato de texto plano estructurado en disco. indent=4 formatea el archivo aplicando espacios estéticos y saltos de línea para que sea legible a simple vista. ensure_ascii=False es crítico para preservar los caracteres especiales del idioma español (como la letra ñ, tildes o símbolos de moneda) en su codificación nativa UTF-8 sin transformarlos a secuencias Unicode complejas.